# Analysis for Lắc Bầu Cua (Vietnamese Lunar New Year game)

Can the right betting strategy survive all 10 rounds and walk away with profit after a session?

In [ ]:
import glob
import pandas as pd
from math import comb
import matplotlib.pyplot as plt

In [18]:
files = sorted(glob.glob("StrategyCSVs_10/*.csv"))
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
display(df.head())

,Session,Strategy_Name,Starting_Bankroll,Max_Rounds,Cost_Per_Round,Rounds_Survived,Final_Bankroll,Bet_Sheet
0,1,1BetAllInAt12Dollars,12.0,10,12.0,10,12.0,FISH:12.0
1,2,1BetAllInAt12Dollars,12.0,10,12.0,10,108.0,FISH:12.0
2,3,1BetAllInAt12Dollars,12.0,10,12.0,6,0.0,FISH:12.0
3,4,1BetAllInAt12Dollars,12.0,10,12.0,1,0.0,FISH:12.0
4,5,1BetAllInAt12Dollars,12.0,10,12.0,1,0.0,FISH:12.0


Clean up strategy labels for consistency

In [19]:
name_map = {
    "1BetAllInAt12Dollars": "All_In_1Bet",
    "2BetAllInAt12Dollars": "All_In_2Bet",
    "3BetAllInAt12Dollars": "All_In_3Bet",
    "4BetAllInAt12Dollars": "All_In_4Bet",
    "5BetAllInAt12Dollars": "All_In_5Bet",
    "AllInAt12Dollars": "All_In_6Bet",
    "Conserve6": "Conservative_6Bet",
    "ConserveBet1": "Conservative_1Bet",
    "ConserveBet2": "Conservative_2Bet",
    "ConserveBet3": "Conservative_3Bet",
    "ConserveBet4": "Conservative_4Bet",
    "ConserveBet5": "Conservative_5Bet",
}

df["Strategy_Name"] = df["Strategy_Name"].map(name_map)

## Strategy Performance Summary

Survival rates, average rounds survived, and profit distributions across all strategies.

In [20]:
df["Success"] = df["Rounds_Survived"] == df["Max_Rounds"]
df["Profit"] = df["Final_Bankroll"] - df["Starting_Bankroll"]

summary = df.groupby("Strategy_Name").agg(
    Success_Rate=("Success", lambda x: round(x.mean() * 100, 1)),
    Avg_Rounds=("Rounds_Survived", lambda x: round(x.mean(), 1)),
    Median_Profit=("Profit", lambda x: round(x.median(), 2)),
    Return_Pct=("Profit", lambda x: round((x.mean() / 12) * 100, 2)),
    Profit_SD=("Profit", lambda x: round(x.std(), 2))
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(summary.reset_index())

,Strategy_Name,Success_Rate,Avg_Rounds,Median_Profit,Return_Pct,Profit_SD
0,All_In_1Bet,17.9,3.5,-12.0,-28.90,21.86
1,All_In_2Bet,17.6,4.1,-12.0,-45.45,13.72
2,All_In_3Bet,13.0,3.2,-4.0,-28.00,9.20
3,All_In_4Bet,14.7,3.8,-6.0,-28.98,5.96
4,All_In_5Bet,9.8,3.1,-2.4,-24.88,3.17
5,Conservative_1Bet,38.9,6.1,-12.0,-21.65,14.86
6,Conservative_2Bet,46.6,7.2,-9.0,-27.72,10.50
7,Conservative_3Bet,47.8,7.3,-8.0,-30.85,7.28
8,Conservative_4Bet,55.8,8.0,-6.0,-32.20,5.53
9,Conservative_5Bet,69.2,8.9,-4.8,-33.24,3.65


## Game Probablity

Probablity of matching dice and the expected value per bet.

In [ ]:
total = 6**3 

probs = []
for matches in range(4):
    binomial_prob = comb(3, matches) * (1/6)**matches * (5/6)**(3-matches)
    probs.append(round(binomial_prob, 4))

ev_df = pd.DataFrame({
    "Matches": [0, 1, 2, 3],
    "Multiplier": [-1, 1, 2, 3],
    "Probability": probs
})

ev_df["Multiplier x Probability"] = ev_df["Multiplier"] * ev_df["Probability"]

display(ev_df)
print(f"\nTotal EV: {ev_df['Multiplier x Probability'].sum() * 100:.2f}%")

[0.5787, 0.3472, 0.0694, 0.0046]


,Matches,Multiplier,Probability,Multiplier x Probability
0,0,-1,0.5787,-0.5787
1,1,1,0.3472,0.3472
2,2,2,0.0694,0.1388
3,3,3,0.0046,0.0138



Total EV: -7.89%
